In [ ]:
import os
import glob
import xarray as xr

In [ ]:
# --- Configuration ---
# Target a specific spinup directory based on your folder structure
data_dir = "/export/lv9/projects/dws/model_output/archived_runs/spinup_10"


# Use wildcard matching to grab all 12 monthly files for the year 2015
file_pattern = os.path.join(data_dir, "dws_500m.3d.2015*.nc")
monthly_files = sorted(glob.glob(file_pattern))
output_file = os.path.join(data_dir, "yearly_dws_500m.3d.2015_N1p_depth_averaged.nc")

# Define the exact list of variables you want to retain in the new file.
# Note: You do not need to list coordinate variables (like time, xc, yc, lonc, latc); 
# xarray will automatically keep any coordinates associated with your selected variables.
variables_to_keep = [
    "bathymetry", 
    "elev", 
    "latc",
    "lonc",
    "N1p"   
]


In [ ]:
# --- Pre-processing Function ---
def clean_and_average(ds):
    """
    1. Subsets variables early to save memory.
    2. Drops overlapping time steps.
    3. Depth-averages any variable that contains a 'level' dimension.
    """
    # 1. Only process the variables we actually care about
    # Keep coordinate variables (time, xc, yc, lonc, latc) automatically
    vars_in_ds = [v for v in variables_to_keep if v in ds]
    ds = ds[vars_in_ds]
    
    # 2. Time Overlap Handling
    fname = os.path.basename(ds.encoding.get("source", ""))
    if "201512" not in fname:
        ds = ds.isel(time=slice(0, -1))
        
    # 3. Depth Averaging for 4D variables
    for var_name in ds.data_vars:
        if 'level' in ds[var_name].dims:
            # Calculate the mean across the water column
            # keep_attrs=True ensures units and long_name are not deleted
            ds[var_name] = ds[var_name].mean(dim='level', keep_attrs=True)
            
            # Optional: Add a note to the metadata that this was averaged
            ds[var_name].attrs['processing'] = 'depth-averaged'

    return ds

In [ ]:
# --- Processing ---
print(f"Found {len(monthly_files)} files. Initiating concatenation...")

# 1. Open and combine all monthly files into a single virtual dataset.
# data_vars="minimal" and coords="minimal" ensure static arrays (bathymetry) aren't duplicated or given time steps.
# compat="override" forces xarray to trust the static mesh geometry from the first file, speeding up loading.
ds = xr.open_mfdataset(
    monthly_files,
    preprocess=clean_and_average,
    combine="by_coords",
    data_vars="minimal",
    coords="minimal",
    compat="override",
    parallel=True # Leverages dask if installed for faster reading
)

# 2. Subset the dataset to keep only the chosen variables
ds_subset = ds[variables_to_keep]

if 'level' in ds_subset.coords:
    ds_subset = ds.drop_vars('level')

# 3. Write the aggregated subset to a new NetCDF file
print(f"Writing combined yearly data to disk at: {output_file}")
print("This may take a few moments depending on the file sizes...")

# Using compute=True (default) evaluates the dask arrays and writes them to the file.
ds_subset.to_netcdf(output_file)

# Free up resources
ds.close()
ds_subset.close()

print("Yearly file successfully created!")